# CPT Notebook A — Chuẩn bị data (Section 6.1)

**Chạy đúng 1 lần, không cần GPU.** Output: dataset đã tokenize + pack 2048 token tại `{HF_USER}/vi-cpt-corpus-2048` (private) — Notebook B (`cpt_b_train.ipynb`) sẽ load thẳng repo này, không phải tokenize lại mỗi session.

Nguồn data (đều public, không gated, parquet native — không dùng CC100/OSCAR vì là dataset kiểu script/gated, `datasets` >= 3.0 không load được):
- `wikimedia/wikipedia` config `20231101.vi` — chất lượng cao
- `HuggingFaceFW/fineweb-2` config `vie_Latn` — web corpus tiếng Việt đã lọc chất lượng, quy mô lớn (stream)
- `HuggingFaceFW/fineweb-edu` `sample-10BT` — English replay ~20% chống catastrophic forgetting

**Trước khi chạy:** Add-ons → Secrets → thêm `HF_TOKEN` (**Write** token — cell Config sẽ tự kiểm tra và báo nếu là Read token). Username HF tự lấy từ token, không cần sửa gì trong notebook.

In [ ]:
!pip install -q -U datasets huggingface_hub pyarrow

In [ ]:
# Config + login — username TỰ LẤY từ token (whoami), không gõ tay để khỏi sai namespace
# (đã từng dính 403 "You don't have the rights to create a dataset under the namespace ..."
#  vì HF_USER gõ tay không khớp username thật của token)
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, whoami
login(UserSecretsClient().get_secret("HF_TOKEN"))

info = whoami()
HF_USER = info["name"]
role = ((info.get("auth") or {}).get("accessToken") or {}).get("role")
assert role != "read", "HF_TOKEN là READ token — tạo token WRITE tại hf.co/settings/tokens rồi cập nhật Kaggle Secret"
print(f"HF user: {HF_USER} (token: {role})")

DATA_REPO = f"{HF_USER}/vi-cpt-corpus-2048"
BLOCK = 2048

In [ ]:
# Tải 3 nguồn — mục tiêu tổng ~400M token (đủ cho max_steps=6000, tỉ lệ vi:en ≈ 80:20)
# QUAN TRỌNG: stream thẳng ra đĩa bằng from_generator, KHÔNG gom vào list Python
# (bản trước dùng Dataset.from_list với 1.5M docs → hết RAM, kernel died)
from datasets import load_dataset, Dataset, DatasetDict, concatenate_datasets

N_WEB_VI = 250_000  # fineweb-2 doc trung bình ~1-2k token → ~250-350M token
N_EN     = 100_000  # ~100M token English replay (~20% tổng)

wiki_vi = load_dataset("wikimedia/wikipedia", "20231101.vi", split="train")  # ~1.3M bài

def stream_texts(repo, config, n, min_len=200):
    def gen():
        ds = load_dataset(repo, config, split="train", streaming=True)
        k = 0
        for r in ds:
            if len(r["text"]) > min_len:
                yield {"text": r["text"]}
                k += 1
                if k >= n:
                    break
    return Dataset.from_generator(gen)  # ghi thẳng Arrow ra đĩa, RAM không phình

web_vi     = stream_texts("HuggingFaceFW/fineweb-2", "vie_Latn", N_WEB_VI)
fineweb_en = stream_texts("HuggingFaceFW/fineweb-edu", "sample-10BT", N_EN)
print(f"wiki_vi={len(wiki_vi)}  web_vi={len(web_vi)}  fineweb_en={len(fineweb_en)}")

In [ ]:
# Tokenize + pack thành block 2048 token (chuẩn CPT: nối tài liệu bằng EOS rồi cắt khúc)
# num_proc=2 (không phải 4): mỗi process ăn thêm RAM, Kaggle CPU dễ chết nếu chạy 4
from transformers import AutoTokenizer
from itertools import chain
tok = AutoTokenizer.from_pretrained("Qwen/Qwen3-1.7B-Base")

def pack(ds):
    def tokenize(batch):
        return {"ids": [tok(t)["input_ids"] + [tok.eos_token_id] for t in batch["text"]]}
    def group(batch):
        flat = list(chain.from_iterable(batch["ids"]))
        n = (len(flat) // BLOCK) * BLOCK
        return {"input_ids": [flat[i:i+BLOCK] for i in range(0, n, BLOCK)]}
    ds = ds.select_columns(["text"]).map(tokenize, batched=True, batch_size=500,
                                         remove_columns=["text"], num_proc=2)
    return ds.map(group, batched=True, batch_size=500, remove_columns=["ids"], num_proc=2)

vi = pack(concatenate_datasets([wiki_vi.select_columns(["text"]),
                                web_vi])).shuffle(seed=42)
en = pack(fineweb_en).shuffle(seed=42)
print(f"packed: vi={len(vi)} block  en={len(en)} block  ({BLOCK} token/block)")

In [ ]:
# Trộn 80/20, tách eval_vi + eval_en (theo dõi catastrophic forgetting), push lên Hub
n_en = min(len(en), int(len(vi) * 0.25))  # 0.25 × vi ≈ 20% tổng
train = concatenate_datasets([vi.select(range(len(vi) - 1000)),
                              en.select(range(n_en - 1000))]).shuffle(seed=42)
DatasetDict({
    "train":   train,
    "eval_vi": vi.select(range(len(vi) - 1000, len(vi))),
    "eval_en": en.select(range(n_en - 1000, n_en)),
}).push_to_hub(DATA_REPO, private=True)
print(f"Xong — data nằm ở {DATA_REPO}. Chuyển sang cpt_b_train.ipynb.")